# 直接使用「現成的情緒模型」：不訓練，拿別人訓練好的模型來評估

## 這份程式要做什麼

這是你這一系列「影評情緒分類」的**第三種做法**：

```
Rotten Tomatoes 的測試集影評（1066 筆）
        │
        ▼  別人已經訓練好的情緒模型（twitter-roberta-base-sentiment-latest）
        │   ※ 我們完全不訓練，直接拿來預測
        ▼
每句影評得到三個分數：negative / neutral / positive
        │
        ▼  只比較 negative 和 positive 兩個分數，取較大的
        ▼
預測：0（負面）或 1（正面）→ 和真實標籤比較，產生分類報告
```

## 三種做法的位置

| 做法 | 做了什麼 | 是否訓練 |
|---|---|---|
| ① 微調 BERT | 用影評資料更新整個模型 | 是（大） |
| ② 凍結 Embedding + 邏輯回歸 | 用向量訓練小分類器 | 是（小） |
| **③ 現成的任務專用模型（這份）** | **直接拿別人訓練好的情緒模型來預測** | **完全不訓練** |

## 什麼是「任務專用模型」（Task-Specific Model）

別人已經針對**情緒分析**這個任務微調好的模型。我們省掉所有訓練步驟，只做兩件事：**載入、預測**。

**這個模型是什麼**：`cardiffnlp/twitter-roberta-base-sentiment-latest` 是 RoBERTa 架構，用大量推特貼文預訓練、再用推特情緒資料微調的。輸出三類：負面、中性、正面。

## 這個做法的兩個「不匹配」

這份程式最值得思考的地方，是模型和資料之間有兩個落差：

| 落差 | 說明 | 影響 |
|---|---|---|
| **領域不同** | 模型看的是**推特貼文**，這裡評估的是**電影影評** | 用語、長度、風格都不同，可能降低表現 |
| **類別數不同** | 模型分**三類**（含中性），資料集只有**兩類**（負面、正面） | 程式用「只比較負面和正面兩個分數」來處理中性 |

這個做法的優點是**零成本**；缺點是**不一定貼合你的資料**。它可以當作「最低成本的基準線」。

## 步驟 0：載入套件

| 匯入 | 用途 |
|---|---|
| `load_dataset` | 載入影評資料集 |
| `pipeline` | 現成的推論工具，把「分詞 → 模型 → 分數」包在一起 |
| `tqdm` | 顯示進度條 |
| `KeyDataset` | 讓 pipeline 逐筆讀取資料集中的某一欄 |
| `numpy` | 用 `argmax` 比較分數 |
| `classification_report` | 產生分類評估報告 |

In [ ]:
#====Task-Specific Model====

from datasets import load_dataset
from transformers import pipeline
from tqdm import tqdm
from transformers.pipelines.pt_utils import KeyDataset
import numpy as np
from sklearn.metrics import classification_report

## 步驟 1：定義評估函式

```python
def evaluate_performance(y_true, y_pred):
    report = classification_report(
        y_true, y_pred, target_names=["negative", "positive"])
    print(report)
```

和上一份完全相同的評估函式：

| 部分 | 說明 |
|---|---|
| `y_true` | 真正的答案（資料集的標籤） |
| `y_pred` | 模型的預測 |
| `target_names` | 把 0、1 換成 `"negative"`、`"positive"`。順序必須對應標籤編號（0 = 負面、1 = 正面） |

**兩種做法用同一個評估函式，結果才能公平比較。**

In [ ]:
def evaluate_performance(y_true, y_pred):
    report = classification_report(
        y_true, y_pred, target_names=["negative", "positive"])
    print(report)

## 步驟 2：載入資料

```python
data = load_dataset("cornell-movie-review-data/rotten_tomatoes")
```

| 欄位 | 說明 |
|---|---|
| `text` | 一句影評 |
| `label` | `0` = 負面、`1` = 正面 |

這份程式**只用測試集**（約 1066 筆，正負各半，我印象中如此，請用 `len(data["test"])` 確認），因為根本不訓練，用不到訓練集。

被註解掉的兩行是用來檢查資料的：

```python
# print(data)                  # 看有哪幾份資料、各有幾筆
# print(data["train"][0])      # 印出第一筆
```

In [ ]:
data = load_dataset("cornell-movie-review-data/rotten_tomatoes")

# print(data)
# print(data["train"][0])  #印出第一筆資料

## 步驟 3：建立 pipeline

```python
model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"

pipe = pipeline(
    model=model_path,
    tokenizer=model_path,
    return_all_scores=True,
    top_k=None,
    device=0,  #使用GPU
)
```

### 這個 pipeline 做了什麼

```
文字 → ① 分詞 → ② 模型（RoBERTa + 分類頭）→ ③ softmax 轉成機率 → 每個類別一個分數
```

### 參數說明

| 參數 | 說明 |
|---|---|
| `model=model_path` | 要載入的模型（會自動下載，之後從快取讀取） |
| `tokenizer=model_path` | 配對的分詞器，用同一個名稱確保配對 |
| `top_k=None` | **回傳所有類別的分數**。預設只回傳分數最高的那一個，設成 `None` 才會全部給 |
| `return_all_scores=True` | 舊版用來「回傳全部分數」的參數，新版已經被 `top_k=None` 取代。兩個一起寫沒有害處，新版可能出現棄用警告 |
| `device=0` | 使用**第 0 張 GPU**。`-1` 代表 CPU |

### 為什麼沒有指定任務類型

一般會寫 `pipeline("text-classification", ...)`。這裡省略了，pipeline 會**依模型的設定自動判斷任務**。

In [ ]:
model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"

pipe = pipeline(
    model=model_path,
    tokenizer=model_path,
    return_all_scores=True,
    top_k=None,
    device=0,  #使用GPU
)

## 步驟 4：查看模型的標籤對應

```python
print(pipe.model.config.id2label)
```

這一行**非常重要**：它告訴你模型輸出的每個類別編號對應什麼意思。我印象中這個模型是：

```
{0: 'negative', 1: 'neutral', 2: 'positive'}
```

### 為什麼要確認

模型的標籤順序**不一定**和你的資料集一樣：

| | 這個模型 | Rotten Tomatoes 資料集 |
|---|---|---|
| 負面 | `negative` | `0` |
| 中性 | `neutral` | **沒有這一類** |
| 正面 | `positive` | `1` |

如果不確認就假設順序，很可能把負面和正面搞反，而且不會報錯，只是分數會很差。

### 被註解掉的測試

```python
# test = pipe("This movie is amazing!", top_k=None)
# print(test)
```

這是用單句測試 pipeline 的寫法。建議打開來看看輸出長什麼樣，下一步會用到這個格式。

In [ ]:
print(pipe.model.config.id2label)     #查看模型的標籤對應index

# test = pipe("This movie is amazing!", top_k=None)
# print(test)  #印出模型對於輸入的預測結果

## 步驟 5：逐筆預測，並把三個分數變成一個決定

這是整份程式**最核心**的一段，先看它想做的事：

```
對測試集的每一句影評：
   ① 讓 pipeline 算出三個類別的分數
   ② 把三個分數整理成 {標籤名稱: 分數} 的字典
   ③ 取出 negative 與 positive 兩個分數，比較大小
   ④ 較大的那個就是預測（0 = 負面、1 = 正面）
```

### `pipe(KeyDataset(data["test"], "text"))`

| 部分 | 說明 |
|---|---|
| `data["test"]` | 測試集 |
| `KeyDataset(..., "text")` | 把資料集包裝成「**逐筆吐出 `text` 欄位**」的物件，讓 pipeline 能逐筆讀取 |
| `pipe(...)` | 對每一筆做預測，並以**生成器**的形式一筆一筆回傳結果 |

用 `KeyDataset` 的好處是：pipeline 可以**串流處理**，在 GPU 上效率較好；如果自己用 `for` 迴圈一句一句呼叫 `pipe(...)`，函式庫可能會警告你「循序使用 pipeline 效率不佳」。

### `tqdm(..., total=len(data["test"]))`

`tqdm` 負責顯示進度條。pipeline 的回傳是**生成器**，沒有長度，所以必須**自己告訴 tqdm 總共有幾筆**（`total=`），進度條才能顯示百分比。

### 每次迴圈拿到的 `output`

`output` 是一個 list，每個元素是一個 dict，包含 `label` 和 `score`。你的註解裡記了一個範例：

```python
[{'label': 'positive', 'score': 0.98},
 {'label': 'neutral',  'score': 0.012},
 {'label': 'negative', 'score': 0.005}]
```

**注意順序**：`top_k=None` 時，結果是**依分數由高到低排序**的（範例中 `positive` 分數最高，所以排第一）。也就是說，**每一筆影評的順序都可能不同**。

### 為什麼要建立 `scores` 字典（這是很聰明的寫法）

```python
for item in output:
    scores[item["label"]] = item["score"]
```

這兩行把「排好序的 list」整理成「用**標籤名稱**查分數」的字典：

```
{'positive': 0.98, 'neutral': 0.012, 'negative': 0.005}
```

之後用 `scores["negative"]`、`scores["positive"]` 取分數，**不管 list 的順序怎麼變，都不會拿錯**。

| 寫法 | 預測 | 說明 |
|---|---|---|
| **用標籤名稱查（你的程式）** | `[1, 0, 0]` | 全部正確 |
| 用固定位置取（`output[0]` 當負面、`output[2]` 當正面） | `[0, 0, 0]` | 第 1 筆錯了：它把排第一的 `positive 0.98` 當成負面分數 |

固定位置的寫法只有在「輸出順序固定是 負面、中性、正面」時才正確（舊版的 `return_all_scores=True` 單獨使用時，我印象中是依標籤編號排列，所以舊教學可以那樣寫）。**在 `top_k=None` 排序輸出的情況下，用標籤名稱查才是對的。**

### 怎麼處理「中性」

```python
negative_score = scores["negative"]
positive_score = scores["positive"]
assigment = np.argmax([negative_score, positive_score])
```

**中性的分數被直接丟掉**，只比較負面和正面。上面第 2 筆影評就是例子：中性分數最高（0.60），但程式只看 negative 0.30 和 positive 0.10，所以仍然預測成負面（0）。

| | 做法 | 後果 |
|---|---|---|
| 現在 | 強迫在負面與正面之間二選一 | 符合資料集只有兩類的事實；但真正「中性」的影評會被硬判成某一邊 |

`np.argmax([負面分數, 正面分數])` 回傳 `0` 或 `1`，正好對應資料集的標籤編號（0 = 負面、1 = 正面），所以可以直接拿去和真實標籤比較。這個編號對應成立的前提，是步驟 4 確認過標籤名稱。

### `y_pred.append(assignment)`

把每一筆的預測存進 `y_pred`，迴圈結束後，`y_pred` 的長度會和測試集一樣，順序也一樣。

### 一個小細節：`scores` 字典是跨迴圈共用的

`scores = {}` 在迴圈外面建立，每一筆都會**覆蓋**同樣的三個 key，所以不會殘留上一筆的分數（因為三個類別每次都會被重新寫入）。但如果某一筆的輸出缺少某個類別，就會悄悄沿用上一筆的舊值。這個模型每次都會給三類，所以沒問題。

### 要跑多久

1066 筆影評，用 GPU 通常是幾十秒（我沒有實測你的環境）。

In [ ]:
y_pred = []

scores = {}

for output in tqdm(pipe(KeyDataset(data["test"], "text")),    #tqdm顯示進度條, total用來顯示bar總數長度
                   total=len(data["test"])):                  #KeyDataset將dataset轉換成可迭代的物件, 只取text欄位
    for item in output:
        scores[item["label"]] = item["score"]                 #output是list, 每個item是dict, 包含label和score
        #[{'label': 'positive', 'score': 0.9824380278587341}, {'label': 'neutral', 'score': 0.012347577139735222}, {'label': 'negative', 'score': 0.005214422941207886}]

    negative_score = scores["negative"]
    positive_score = scores["positive"]

    assignment = np.argmax([negative_score, positive_score])
    y_pred.append(assignment)

## 步驟 6：評估

```python
evaluate_performance(data["test"]["label"], y_pred)
```

**注意參數順序：第一個是真正的答案，第二個是預測。**

### 怎麼讀報告

報告格式和上一份相同：`precision`（模型說是這一類的，有多少真的是）、`recall`（真正是這一類的，模型找出多少）、`f1-score`（兩者的調和平均）、`support`（該類在測試集的真實筆數）。

我用上面那三筆假資料實測，格式如下（只是示範格式，不是你的結果）：

```
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00         2
    positive       1.00      1.00      1.00         1

    accuracy                           1.00         3
```

### 預期的表現，與如何比較


比較三種做法時，可以這樣思考：

| 如果結果是… | 可能的原因 |
|---|---|
| 比凍結 Embedding + 邏輯回歸**低** | 模型的訓練領域（推特）和資料（影評）有落差；中性的影評被強迫二選一 |
| 比微調 BERT **低** | 微調直接針對影評調整，這個模型沒看過影評 |
| 都還不錯 | 情緒是比較通用的概念，跨領域也能遷移 |

這個做法**完全不需要訓練資料與時間**，是最省成本的基準線。

In [ ]:
evaluate_performance(data["test"]["label"], y_pred)

---

# 重點整理

## 一、整份程式一句話

> 直接載入別人在推特資料上訓練好的三類情緒模型，用 pipeline 對 Rotten Tomatoes 的測試集逐筆預測，丟掉中性分數，只比較負面與正面兩個分數取較大者，再用分類報告評估，全程不做任何訓練。

## 二、整條流程（必須搞懂）

| 步驟 | 程式 | 結果 |
|---|---|---|
| 載入資料 | `load_dataset(...)` | 只用測試集 |
| 建立 pipeline | `pipeline(model=..., top_k=None, device=0)` | 能對文字輸出所有類別的分數 |
| 確認標籤對應 | `pipe.model.config.id2label` | 知道 0 / 1 / 2 各代表什麼 |
| 逐筆預測 | `pipe(KeyDataset(data["test"], "text"))` | 每筆得到三個分數 |
| 整理成字典 | `scores[item["label"]] = item["score"]` | 用名稱查分數，不受順序影響 |
| 二選一 | `np.argmax([negative_score, positive_score])` | 0 或 1，丟掉中性 |
| 評估 | `evaluate_performance(...)` | 分類報告 |

## 三、必須搞懂的 10 個概念

| # | 概念 | 一句話解釋 |
|---|---|---|
| 1 | 任務專用模型 | 別人已針對某個任務微調好的模型，可直接使用 |
| 2 | 領域遷移 | 模型在一個領域訓練，拿到另一個領域使用（推特 → 影評） |
| 3 | `pipeline` | 把分詞、模型、後處理包成一個工具 |
| 4 | `top_k=None` | 回傳所有類別的分數，而不只是最高的 |
| 5 | 輸出排序 | `top_k=None` 的結果依分數由高到低排，每筆順序可能不同 |
| 6 | `id2label` | 模型輸出編號與類別名稱的對應 |
| 7 | `KeyDataset` | 讓 pipeline 逐筆讀取資料集的某一欄 |
| 8 | `tqdm` 的 `total` | 生成器沒有長度，必須自己告訴進度條總數 |
| 9 | 用名稱查分數 | 比用固定位置取分數穩妥 |
| 10 | 丟掉中性 | 三類模型用在二類資料時的處理方式 |

## 四、最容易搞混的地方

1. **這份有訓練嗎？**
   完全沒有。只載入別人訓練好的模型並預測。

2. **為什麼不直接用 `output[0]`、`output[2]`？**
   `top_k=None` 會依分數排序，順序每筆都不同，用固定位置會拿錯分數。這份用標籤名稱查，是對的。

3. **為什麼要看 `id2label`？**
   確認每個編號代表什麼，避免把負面和正面搞反。

4. **三類模型怎麼用在二類資料？**
   丟掉中性分數，只比較負面和正面。

5. **預測值 0 和 1 為什麼能直接和標籤比較？**
   因為 `np.argmax([負面, 正面])` 回傳的編號，剛好對應資料集的 0 = 負面、1 = 正面。

6. **為什麼進度條需要 `total`？**
   pipeline 回傳生成器，沒有長度資訊。

7. **這個做法算「零樣本」嗎？**
   不嚴格。模型本來就針對情緒任務訓練過，只是沒看過這份資料（影評），所以更貼切的說法是「領域遷移」。

## 五、常見錯誤排查表

| 症狀 | 可能原因 | 處理 |
|---|---|---|
| `KeyError: 'negative'` | 這個模型的標籤名稱和預期不同 | 先 `print(pipe.model.config.id2label)`，依實際名稱修改 |
| 分數很差，近乎隨機 | 標籤順序搞反，或順序假設錯誤 | 確認 `id2label`；用標籤名稱而不是固定位置 |
| 預測長度和標籤長度不一致 | `y_pred.append` 放錯縮排位置（例如放到內層迴圈裡） | 確認 `append` 在外層迴圈、內層迴圈之外 |
| `CUDA` 相關錯誤、找不到 `device=0` | 沒有 GPU | 改成 `device=-1`（用 CPU，會慢一些） |
| 警告 `return_all_scores` 已棄用 | 新版以 `top_k=None` 取代 | 只保留 `top_k=None` |
| 進度條不顯示百分比 | 沒有 `total=` | 加上 `total=len(data["test"])` |
| 警告 pipeline 循序使用效率不佳 | 自己用 `for` 一句句呼叫 `pipe` | 改用 `KeyDataset` |

## 六、觀察到的幾個地方（程式碼未修改，僅供參考）

1. **兩個匯入沒用到**
   `AutoTokenizer`、`AutoModelForSequenceClassification`。

2. **`return_all_scores=True` 與 `top_k=None` 重複**
   新版以後者取代前者，前者可能造成警告。

3. **中性被硬判成負面或正面**
   真正的中性影評會被強迫二選一，可能拉低分數。

4. **領域不一致**
   模型針對推特訓練，影評的用語、長度不同。

5. **預測的是 `np.int64`**
   `y_pred` 裡是 numpy 整數，交給 `classification_report` 沒有問題。

6. **`scores` 字典跨迴圈共用**
   如果某筆缺少某個類別，會沿用上一筆的舊值；這個模型每筆都給三類，所以沒問題。

7. **兩個筆誤**
   `Tesk`（應為 `Task`）、`assigment`（應為 `assignment`），都不影響執行。

8. **沒有儲存預測結果**
   跑完就消失了，想比較不同做法時需要自己存。

9. **只有單次評估**
   沒有信賴區間或多次比較。

## 七、可以自己做的實驗（需要時再自己加，不影響原程式）

```python
# 1. 看單句的輸出格式
print(pipe("This movie is amazing!"))
print(pipe("A dull and lifeless film."))

# 2. 看模型的標籤對應
print(pipe.model.config.id2label)

# 3. 統計「中性分數最高」的影評有多少（也就是被強迫二選一的比例）
# 需要在迴圈裡另外記錄 scores["neutral"] 是否為三者最大

# 4. 看分錯的影評，找出模型的弱點
true = data["test"]["label"]
wrong = [i for i, (t, p) in enumerate(zip(true, y_pred)) if t != p]
for i in wrong[:5]:
    print(true[i], y_pred[i], data["test"]["text"][i])

# 5. 換一個模型比較（例如針對電影影評或一般文字訓練的情緒模型）
# model_path = "別的情緒模型的名稱"

# 6. 把預測結果存起來
import numpy as np
np.save("y_pred_task_specific.npy", np.array(y_pred))

# 7. 沒有 GPU 時改用 CPU
# pipe = pipeline(model=model_path, tokenizer=model_path, top_k=None, device=-1)

# 8. 三種做法並排比較：微調 BERT / 凍結 Embedding + 邏輯回歸 / 這份
```

## 八、三種影評情緒分類做法的對照

| | ① 微調 BERT | ② 凍結 Embedding + 邏輯回歸 | **③ 現成任務模型（這份）** |
|---|---|---|---|
| 需要訓練資料 | 要（幾千筆） | 要（幾千筆） | **不要** |
| 訓練成本 | 高 | 低 | **無** |
| 模型針對你的資料調整 | 是 | 部分（分類器） | **否** |
| 實作複雜度 | 高 | 低 | **最低** |
| 效果（一般） | 最好 | 不錯 | 視領域與類別是否匹配 |
| 最大風險 | 需要資源與調參 | 向量不針對任務 | **領域不匹配、類別數不匹配** |

## 九、怎麼選

| 你的情況 | 建議 |
|---|---|
| 沒有任何標註資料、想先看看效果 | **這份：直接用現成模型** |
| 有少量標註資料、資源有限 | 凍結 Embedding + 邏輯回歸 |
| 標註資料很少（幾十筆） | SetFit |
| 資料充足、追求最佳效果 | 微調整個模型 |
| 任務很特殊，找不到現成模型 | 微調，或用生成式模型加提示詞 |

## 十、一句話記住

> **能直接用別人訓練好的模型，就不要從頭訓練。** 但要檢查三件事：標籤的對應（`id2label`）、輸出的順序（用名稱查，不要用位置）、模型的領域與類別數是否和你的資料匹配。